In [ ]:
# ============================================================
# 04_lift_highrisk_costbenefit.ipynb
# Lift-based high-risk segmentation and scenario-based actuarial
# cost-benefit analysis, extended with a probabilistic
# (Monte Carlo) sensitivity analysis of the loss-ratio impact.
#
# Input : data/processed/scored_male.csv, scored_female.csv
# Output: data/processed/highrisk_nondiabetic_{sex}.csv,
#         results/tables/lift_summary.csv,
#         results/tables/cost_benefit_scenarios.csv,
#         results/tables/cost_benefit_montecarlo.csv,
#         results/figures/lift_curves_male_female.(png|pdf),
#         results/figures/costbenefit_montecarlo.(png|pdf),
#         results/figures/costbenefit_tornado.(png|pdf)
#
# Figures: grayscale (seaborn), dpi 600, PNG + PDF, no titles/captions.
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

SEED = 42
rng = np.random.default_rng(SEED)

ROOT = Path.cwd()
while not (ROOT / "data").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"
FIG = ROOT / "results" / "figures"
TAB = ROOT / "results" / "tables"
for p in (FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)

GRAY = ["#000000", "#595959", "#8C8C8C", "#BFBFBF", "#D9D9D9"]
sns.set_theme(style="whitegrid", context="paper")
mpl.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.family": "serif", "font.size": 11, "axes.edgecolor": "black",
    "axes.linewidth": 0.8, "axes.grid": True, "grid.color": "0.88", "grid.linewidth": 0.6,
})
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

TARGET = "HE_DM_HbA1c"
LIFT_THRESHOLD = 2.0

# ---- Lift analysis and high-risk identification ----
lift_rows, lift_curves = [], {}
for sex in ["male", "female"]:
    d = pd.read_csv(PROC / f"scored_{sex}.csv").sort_values("risk_prob", ascending=False).reset_index(drop=True)
    n, prev = len(d), d[TARGET].mean()
    pct = np.arange(1, n + 1) / n
    cum_pos_rate = d[TARGET].cumsum().values / np.arange(1, n + 1)
    cum_lift = cum_pos_rate / prev
    lift_curves[sex] = (pct, cum_lift)
    # top fraction where cumulative lift >= 2 : report the top-40% boundary
    k = int(round(0.40 * n))
    lift_at_40 = cum_lift[k - 1]
    highrisk = d.iloc[:k].copy()
    nd_highrisk = highrisk[highrisk[TARGET] == 0].copy()   # non-diabetic high-risk
    nd_highrisk.to_csv(PROC / f"highrisk_nondiabetic_{sex}.csv", index=False)
    lift_rows.append(dict(Sex=sex, N=n, Prevalence=round(prev, 3),
                          Top40_n=k, CumLift_top40=round(lift_at_40, 3),
                          NonDiab_highrisk_n=len(nd_highrisk)))
    print(f"{sex:7s}: N={n}, top40% n={k}, cum-lift={lift_at_40:.3f}, "
          f"non-diabetic high-risk n={len(nd_highrisk)}")
lift_df = pd.DataFrame(lift_rows)
lift_df.to_csv(TAB / "lift_summary.csv", index=False)

# ---- Figure: cumulative lift curves ----
fig, ax = plt.subplots(figsize=(5.2, 4.2))
for sex, gc, ls in [("male", GRAY[0], "-"), ("female", GRAY[2], "--")]:
    pct, cl = lift_curves[sex]
    ax.plot(pct * 100, cl, color=gc, linestyle=ls, lw=1.8, label=sex.capitalize())
ax.axhline(LIFT_THRESHOLD, color=GRAY[1], lw=1.0, linestyle=":")
ax.axvline(40, color=GRAY[3], lw=1.0, linestyle=":")
ax.set_xlabel("Top percentage of predicted-risk distribution (%)")
ax.set_ylabel("Cumulative lift")
ax.legend(frameon=False, fontsize=9)
save_fig(fig, "lift_curves_male_female")

# ============================================================
# Scenario-based actuarial cost-benefit (per 1,000 screened)
# Net prevented claims ($K) = 1000 * pi * eps * dL / 1000
# Screening cost ($K)       = 1000 * C_screen / 1000
# ============================================================
C_SCREEN = 50.0           # USD per applicant
PREMIUM_BASE = 1200.0     # USD annual premium per policyholder
BASE_LOSS_RATIO = 0.75
HORIZON_YEARS = 10        # simulation window for the screened high-risk cohort
# Loss-ratio improvement (pp) = averted claims over the window, expressed
# as a share of the screened cohort's premium accrued over the same window.
LR_DENOM_K = 1000 * PREMIUM_BASE * HORIZON_YEARS / 1000  # premium ($K) per 1,000 over horizon

scenarios = [
    dict(Scenario="Conservative", pi=0.10, eps=0.15, dL=8000),
    dict(Scenario="Base",         pi=0.15, eps=0.20, dL=10000),
    dict(Scenario="Optimistic",   pi=0.20, eps=0.25, dL=12000),
]
rows = []
for s in scenarios:
    screen_k = 1000 * C_SCREEN / 1000
    prevented_k = 1000 * s["pi"] * s["eps"] * s["dL"] / 1000
    net_k = prevented_k - screen_k
    lr_improve_pp = prevented_k / LR_DENOM_K * 100  # pp over the horizon
    rows.append(dict(**s, Screening_cost_K=round(screen_k, 1),
                     Prevented_claims_K=round(prevented_k, 1),
                     Net_benefit_K=round(net_k, 1),
                     LossRatio_improvement_pp=round(lr_improve_pp, 2)))
cb = pd.DataFrame(rows)
cb.to_csv(TAB / "cost_benefit_scenarios.csv", index=False)
print("\nScenario cost-benefit:\n", cb[["Scenario", "Net_benefit_K", "LossRatio_improvement_pp"]].to_string(index=False))

# ============================================================
# Probabilistic (Monte Carlo) sensitivity analysis  [NEW]
# Parameter uncertainty: pi, eps ~ Beta; dL ~ triangular.
# ============================================================
N_SIM = 50000
def beta_from_range(lo, hi):
    # Beta parameters giving mean at midpoint, moderate spread within [lo,hi]
    mean = (lo + hi) / 2
    a, b = 4.0, 4.0 * (1 - mean) / mean
    return a, b, lo, hi

pi_a, pi_b, pi_lo, pi_hi = beta_from_range(0.10, 0.20)
ep_a, ep_b, ep_lo, ep_hi = beta_from_range(0.15, 0.30)
pi_s = pi_lo + (pi_hi - pi_lo) * rng.beta(pi_a, pi_b, N_SIM)
ep_s = ep_lo + (ep_hi - ep_lo) * rng.beta(ep_a, ep_b, N_SIM)
dL_s = rng.triangular(8000, 10000, 12000, N_SIM)

prevented_k = 1000 * pi_s * ep_s * dL_s / 1000
screen_k = 1000 * C_SCREEN / 1000
net_k = prevented_k - screen_k
lr_pp = prevented_k / LR_DENOM_K * 100

mc = dict(
    metric=["Net_benefit_K", "LossRatio_improvement_pp"],
    mean=[net_k.mean(), lr_pp.mean()],
    median=[np.median(net_k), np.median(lr_pp)],
    p2_5=[np.percentile(net_k, 2.5), np.percentile(lr_pp, 2.5)],
    p97_5=[np.percentile(net_k, 97.5), np.percentile(lr_pp, 97.5)],
    prob_positive=[(net_k > 0).mean(), (lr_pp > 0).mean()],
)
mc_df = pd.DataFrame(mc).round(3)
mc_df.to_csv(TAB / "cost_benefit_montecarlo.csv", index=False)
print("\nMonte Carlo sensitivity (50,000 draws):\n", mc_df.to_string(index=False))
print(f"P(net benefit > 0) = {(net_k > 0).mean()*100:.1f}%")

# ---- Figure: Monte Carlo distributions (net benefit & loss-ratio pp) ----
fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, data, lab in [(axes[0], net_k, "Net benefit per 1,000 screened ($K)"),
                      (axes[1], lr_pp, "Loss-ratio improvement (pp)")]:
    ax.hist(data, bins=60, color=GRAY[3], edgecolor=GRAY[1], linewidth=0.4)
    ax.axvline(np.median(data), color=GRAY[0], lw=1.6, label="Median")
    ax.axvline(np.percentile(data, 2.5), color=GRAY[1], lw=1.0, linestyle="--")
    ax.axvline(np.percentile(data, 97.5), color=GRAY[1], lw=1.0, linestyle="--", label="95% interval")
    ax.set_xlabel(lab); ax.set_ylabel("Frequency")
    ax.legend(frameon=False, fontsize=8)
save_fig(fig, "costbenefit_montecarlo")

# ---- Figure: tornado (one-at-a-time sensitivity of net benefit) ----
base = dict(pi=0.15, eps=0.225, dL=10000)
def net_of(pi, eps, dL):
    return (1000 * pi * eps * dL / 1000) - screen_k
ranges = dict(pi=(0.10, 0.20), eps=(0.15, 0.30), dL=(8000, 12000))
labels = {"pi": r"Uptake $\pi$", "eps": r"Efficacy $\varepsilon$", "dL": r"Averted claims $\Delta L$"}
base_net = net_of(**base)
tor = []
for k, (lo, hi) in ranges.items():
    low = net_of(**{**base, k: lo}); high = net_of(**{**base, k: hi})
    tor.append((labels[k], low - base_net, high - base_net, abs(high - low)))
tor.sort(key=lambda r: r[3])
fig, ax = plt.subplots(figsize=(5.4, 3.0))
for i, (lab, lo, hi, _) in enumerate(tor):
    ax.barh(i, hi - lo, left=min(lo, hi), color=GRAY[3], edgecolor="black", height=0.6)
ax.axvline(0, color=GRAY[0], lw=1.0)
ax.set_yticks(range(len(tor))); ax.set_yticklabels([t[0] for t in tor])
ax.set_xlabel("Change in net benefit vs base ($K per 1,000)")
save_fig(fig, "costbenefit_tornado")
print("\nFigures written: lift_curves, costbenefit_montecarlo, costbenefit_tornado (png+pdf).")
